In [ ]:
from ase.io import read, write
from ase.units import fs, kB
from ase.md import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md import MDLogger
import os

In [ ]:
from mace.calculators import mace_off
calc = mace_off(model="small",device="cpu")

In [ ]:
#create system
from ase.build import molecule
atoms = molecule('H2O', vacuum=3.0)
print(atoms)

In [ ]:
# create function that would run a SP CP2K calculation (needs to run without arguments, can be done as done in the past)
from pycp2k.templates.GLOBAL.GLOBAL import CP2K
from pycp2k.templates.FORCE_EVAL.xTB_templates import add_xTB_OT
from pycp2k.templates.PRINT.singlepoint import *
def return_cp2k_singlepoint(atoms,dyn,dask_cluster=None):
    def cp2k_singlepoint():
        calc=CP2K(cp2k_command="cp2k.psmp", #replace this with your actual cp2k commmand (e.g. "mpirun -np 4 cp2k.psmp")
          working_directory=os.getcwd(),
          project_name=f"snap_{dyn.get_time()/(1000 * fs)}", #Project files will have this base name
          run_type="ENERGY_FORCE",
          print_level="MEDIUM")
        add_xTB_OT(atoms=atoms, calc=calc,
           feval_idx=0,
           preconditioner="FULL_ALL",minimizer="DIIS", # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.OT.add_OT
           scf_guess="RESTART",max_scf=20,eps_scf=1e-6, # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.loop.add_inner_SCF
           outer_max_scf=2,outer_eps_scf=1e-6, # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.loop.add_outer_SCF
           #FIXME: Add option to run with non-cubic cells and no PBC # Options for pycp2k.templates.FORCE_EVAL.SUBSYS.add_atoms.add_cell
           )
        forces_path=add_print_singlepoint_forces(calc=calc,filename="forces",unit="EV/ANGSTROM")
        stress_path=add_print_stress_tensor(calc=calc,filename="./",unit="EV/ANGSTROM^3")
        calc.run()
        atoms.info["E"]=postprocess_energy(calc=calc)
        atoms.set_array("forces",postprocess_forces(forces_path=forces_path))
        
    return cp2k_singlepoint



In [ ]:
# Create the Dask function that wil take run_cp2k (get code from pycp2k examples on ARCEHR2)
# Dask needs to work in a way that the MACE calculation does not stop, but at the end of it it makes sure all submitted jobs are done or crashed

In [ ]:
#define print_log and print_snapshot
def return_print_log(dyn):
    def print_log():
        print(f"Time: {dyn.get_time()/(1000 * fs)} ps")
    return print_log

def return_print_snapshot(atoms,basename):
    def print_snapshot():
        #atoms.arrays["forces"]=atoms.get_forces()
        atoms.arrays['node_energy']=atoms.calc.results['node_energy']
        atoms.center()
        write(f"{basename}_trj.xyz",atoms,format="extxyz",append=True)
    return print_snapshot
# attach the MD runner
atoms.calc=calc
dyn=Langevin(atoms=atoms, timestep=1*fs, temperature_K=300, friction=0.01)
print_log=return_print_log(dyn)
print_snapshot=return_print_snapshot(atoms,basename="H2O")
cp2k_singlepoint=return_cp2k_singlepoint(atoms,dyn)
dyn.attach(print_log, interval=100)
dyn.attach(print_snapshot, interval=100)
dyn.attach(cp2k_singlepoint, interval=100)
dyn.run(1000)


In [ ]:
# Attach the CP2K function to the MD runner

In [ ]:
# Run the MD